In [1]:
!git lfs install
!git clone https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507 ./localqwen

Git LFS initialized.
Cloning into './localqwen'...
remote: Enumerating objects: 25, done.
remote: Total 25 (delta 0), reused 0 (delta 0), pack-reused 25 (from 1)
Receiving objects: 100% (25/25), 1.73 MiB | 19.66 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Filtering content: 100% (4/4), 7.50 GiB | 42.30 MiB/s, done.


In [2]:
!gdown 1X7MjI27W5mB5dxAeJPaOkFgnG1zXKEnW
!unzip ./ml-olympiad-red-task.zip

Downloading...
From (original): https://drive.google.com/uc?id=1X7MjI27W5mB5dxAeJPaOkFgnG1zXKEnW
From (redirected): https://drive.google.com/uc?id=1X7MjI27W5mB5dxAeJPaOkFgnG1zXKEnW&confirm=t&uuid=e813cdbb-0742-4387-86ba-47028c54eb51
To: /content/ml-olympiad-red-task.zip
100% 128M/128M [00:01<00:00, 99.9MB/s]
Archive:  ./ml-olympiad-red-task.zip
   creating: ml-olympiad-red-task/
  inflating: __MACOSX/._ml-olympiad-red-task  
   creating: ml-olympiad-red-task/metrics/
  inflating: __MACOSX/ml-olympiad-red-task/._metrics  
  inflating: ml-olympiad-red-task/УСЛОВИЯ.pdf  
  inflating: __MACOSX/ml-olympiad-red-task/._УСЛОВИЯ.pdf  
  inflating: ml-olympiad-red-task/.DS_Store  
  inflating: __MACOSX/ml-olympiad-red-task/._.DS_Store  
  inflating: ml-olympiad-red-task/УСЛОВИЯ.docx  
  inflating: __MACOSX/ml-olympiad-red-task/._УСЛОВИЯ.docx  
   creating: ml-olympiad-red-task/data/
  inflating: __MACOSX/ml-olympiad-red-task/._data  
  inflating: ml-olympiad-red-task/УСЛОВИЯ.md  
  inflating: __

In [3]:
!pip install -q peft trl datasets bitsandbytes scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.1 MB/s eta 0:00:00


In [4]:
import os
import random
import numpy as np
import torch
import pickle
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
model_name = "./localqwen"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

lora_settings = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

train_model = get_peft_model(base_model, lora_settings)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [6]:
raw_data = load_dataset("json", data_files="ml-olympiad-red-task/data/kid_adult.jsonl", split="train")

def format(data):
    conversation = [
        {"role": "user", "content": data["prompt"]},
        {"role": "assistant", "content": data["kid"]}
    ]
    formatted_text = tokenizer.apply_chat_template(conversation, tokenize=False)
    return {"formatted_text": formatted_text}

train_data = raw_data.map(
    format,
    remove_columns=raw_data.column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [7]:
train_args = SFTConfig(
    output_dir="./sft_train",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2.5e-4,
    max_steps=30,
    seed=42,
    dataset_text_field="formatted_text"
)

sft_trainer = SFTTrainer(
    model=train_model,
    train_dataset=train_data,
    args=train_args,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [ ]:
sft_trainer.train()

train_model.save_pretrained("./sft_model")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss


In [ ]:
test_data = load_dataset("json", data_files="ml-olympiad-red-task/data/public_test_style.jsonl", split="train")

model.eval()
generated_responses = []

with torch.no_grad():
    for row in tqdm(test_data):
        prompt_messages = [{"role": "user", "content": row['prompt']}]

        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

        outputs = train_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

        answer_only = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        generated_responses.append(answer_only)

In [ ]:
with open("ml-olympiad-red-task/metrics/style_clf.pkl", "rb") as f:
    style_evaluator = pickle.load(f)

In [ ]:
features = []

for vec in style_evaluator['vecs']:
    transformed_data = vec.transform(test_answers)
    features.append(transformed_data)

if len(features) > 1:
    vectorized_answers = sp.hstack(features)
else:
    vectorized_answers = features[0]

simple_prob = style_evaluator['clf'].predict_proba(vectorized_answers)[:, 1]

mean_p = np.mean(simple_prob)

print(f"P_simple: {mean_p}")